<a href="https://colab.research.google.com/github/claudiodanielpc-ag/siimag/blob/main/boletin_siimag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Instalar librería
!pip install git+https://github.com/claudiodanielpc-ag/cd_base.git
!pip uninstall -y paramiko sshtunnel
!pip install paramiko==2.11.0 sshtunnel==0.4.0
from cd_base import ConexionBD

##Montar drive
from google.colab import drive
drive.mount('/content/drive')

#Librerías para análisis o adicionales
import pandas as pd
import re
!pip install unidecode
import unidecode
import base64
#Gemini
from google.colab import ai


#librerías para envío de correo
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText



  Cloning https://github.com/claudiodanielpc-ag/cd_base.git to /tmp/pip-req-build-0zm5n_1w
  Running command git clone --filter=blob:none --quiet https://github.com/claudiodanielpc-ag/cd_base.git /tmp/pip-req-build-0zm5n_1w
  Resolved https://github.com/claudiodanielpc-ag/cd_base.git to commit 4e555d17f99aaec598326f185454ff183efabf26
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.9/223.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.3 MB/s eta 0:00:00
  Created wheel for cd_base: filename=cd_base-1.0.0-py3-none-any.whl size=3363 sha256=67881ff62e39f45edcff8687c056b408dae18a6babe62

/usr/local/lib/python3.12/dist-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/usr/local/lib/python3.12/dist-packages/paramiko/transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 6.0 MB/s eta 0:00:00


In [ ]:
ruta = "/content/drive/MyDrive/credenciales/bd_produccion.txt"

bd = ConexionBD(ruta)
engine = bd.conectar("migracion_aws_do")

✅ Conectado a base: migracion_aws_do


In [ ]:
def clean_names(df):
    df = df.copy()
    new_cols = []

    for col in df.columns:
        col = str(col)
        col = col.strip()                   # quitar espacios al inicio/fin
        col = col.lower()                   # minúsculas
        col = unidecode.unidecode(col)      # quitar acentos
        col = re.sub(r"[^\w\s]", "", col)   # quitar caracteres raros
        col = re.sub(r"\s+", "_", col)      # espacios → _
        new_cols.append(col)

    df.columns = new_cols
    return df

In [ ]:
conn = engine.raw_connection()
cursor = conn.cursor()

cursor.callproc(
    'sp_tbl_indicadores_general_programas',
    (1, None, None, '2026-01-01', '2026-03-31', '2025-01-01', '2025-03-31')
)

resultados = []

while True:
    rows = cursor.fetchall()

    if rows:
        cols = [col[0] for col in cursor.description]
        df = pd.DataFrame(rows, columns=cols)
        resultados.append(df)

    if not cursor.nextset():
        break

cursor.close()

In [ ]:
indicadores=resultados[0]
indicadores

,id_indicador,nombre,porcentual,categoria,total,total_comp,porcentaje,color_sube,color_baja,descripcion,formula,tendencia
0,3,Inscritos,0,1,570.00,641.000000,-11.08,#388e3c,#f44336,Alumnos dados de alta en el programa académico.,None,down
1,4,Inscritos facturados,0,1,558.00,542.000000,2.95,#388e3c,#f44336,Alumnos inscritos que ya han cargado al menos ...,None,up
2,6,Cargas,0,1,15854.00,16854.000000,-5.93,#388e3c,#f44336,Materias que los alumnos inscriben durante un ...,None,down
3,9,Eficiencia terminal,1,1,30.37,25.510000,19.05,#388e3c,#f44336,Porcentaje de estudiantes que ya concluyeron ...,Alumnos que Finalizaron Materias / Alumnos que...,up
4,5,Reactivados,0,2,137.00,321.000000,-57.32,#388e3c,#f44336,Alumnos que interrumpieron temporalmente su tr...,None,down
5,12,Tasa de reactivación,1,2,63.72,126.879997,-49.78,#388e3c,#f44336,Porcentaje de estudiantes que deciden retomar ...,Reactivados / (Bajas de la empresa + suspendid...,down
6,10,Reinscritos,0,2,158.00,160.000000,-1.25,#388e3c,#f44336,Alumnos que interrumpieron temporalmente su tr...,None,down
7,7,Tasa de reinserción,1,2,31.35,30.360001,3.26,#388e3c,#f44336,Porcentaje en que los estudiantes vuelven a in...,Reinscritos/ bajas de empresa,up
8,1,Bajas de la empresa,0,3,504.00,527.000000,-4.36,#f44336,#388e3c,"Alumnos que dejan de laborar en la empresa y, ...",None,down
9,2,Bajas del programa,0,3,202.00,187.000000,8.02,#f44336,#388e3c,Alumnos que dejan el programa académico pero c...,None,up


In [ ]:
acumulado=indicadores[["nombre","total","total_comp","porcentaje"]]
#Renombrar total como "2025" y total_comp como "2026"
acumulado=acumulado.rename(columns={"total":"enero-marzo 2026","total_comp":"enero-marzo 2025","porcentaje":"Variación porcentual"})
#Dejar solo Inscritos, Inscritos facturados cargas, eficiencia terminal y tasa de deserción
principales = [
    "Inscritos",
    "Inscritos facturados",
    "Cargas",
    "Eficiencia terminal",
    "Tasa de Deserción"
]

acumulado = acumulado[acumulado["nombre"].isin(principales)]

# lista de variables sin decimales
sin_decimales = ["Inscritos", "Inscritos facturados", "Cargas"]

def formatear_valor(row, col):
    if row["nombre"] in sin_decimales:
        return f"{int(row[col]):,}"
    else:
        return f"{row[col]:,.2f}"

# aplicar formato por columna
for col in ["enero-marzo 2026", "enero-marzo 2025"]:
    acumulado[col] = acumulado.apply(lambda row: formatear_valor(row, col), axis=1)

# variación porcentual (siempre con decimales)
acumulado["Variación porcentual"] = acumulado["Variación porcentual"].apply(lambda x: f"{x:.2f}%")
acumulado = acumulado.replace("Cargas", "Cargas de materias")
#Tabla a markdown
tabla_analisis = acumulado.to_markdown(index=False)
#Reemplazar "Cargas" por "Cargas de materias"

acumulado

,nombre,enero-marzo 2026,enero-marzo 2025,Variación porcentual
0,Inscritos,570,641,-11.08%
1,Inscritos facturados,558,542,2.95%
2,Cargas de materias,"15,854","16,854",-5.93%
3,Eficiencia terminal,30.37,25.51,19.05%
10,Tasa de Deserción,36.20,34.50,4.93%


In [ ]:
# 2. Creamos el prompt con la tabla integrada
instruccion = f"""
Actúa como un analista de datos. Devuelve un texto breve con un análisis de la siguiente tabla de indicadores académicos:

{tabla_analisis}

Menciona los principales cambios entre enero-marzo 2025 y enero-marzo 2026. Siempre has referencia
No inventes datos y utiliza un lenguaje formal y sin valoraciones.
No lances ninguna conclusión. Solo los datos crudos.

El texto debe empezar con lo siguiente: 'En enero-marzo 2026, los principales indicadores académicos tuvieron el siguiente comportamiento:...'
"""

# 3. Enviamos la consulta y GUARDAMOS el resultado en una variable
# Nota: ai.chat devuelve un objeto, usamos .text para extraer el string
analisis_final = ai.generate_text(instruccion)
def format_miles(texto):
    def reemplazo(match):
        num = match.group()

        # Evitar años (1900–2099)
        if 1900 <= int(num) <= 2099:
            return num

        return f"{int(num):,}"

    return re.sub(r'\d{4,}', reemplazo, texto)

analisis_final = format_miles(analisis_final)
# 4. Ahora ya tienes el texto guardado en 'analisis_final'
print("--- Análisis Generado ---")
print(analisis_final)

--- Análisis Generado ---
En enero-marzo 2026, los principales indicadores académicos tuvieron el siguiente comportamiento:

Los Inscritos disminuyeron en un 11.08%, pasando de 641 en enero-marzo 2025 a 570 en enero-marzo 2026. Los Inscritos facturados experimentaron un aumento del 2.95%, ascendiendo de 542 a 558. Las Cargas de materias mostraron una reducción del 5.93%, pasando de 16,854 a 15,854. La Eficiencia terminal se incrementó en un 19.05%, situándose en 30.37% frente al 25.51% del periodo anterior. Finalmente, la Tasa de Deserción registró un aumento del 4.93%, pasando del 34.50% al 36.20%.


In [ ]:
texto="Durante el trimestre enero-marzo de 2026, los principales indicadores de desempeño tuvieron el siguiente comportamiento con respecto al mismo período de 2025:<br>*El número de inscritos disminuyó en 11.08% al pasar de 641 a 570.<br>Las cargas de materias registraron una reducción de 5.93% al disminuir de 16,854 a 15,854.<br><br>La <b>eficiencia terminal</b> tuvo un aumento significativo del 19.05%, pasando del 25.51% al 30.37%.<br><br>Finalmente, la Tasa de Deserción incrementó en 4.93% al pasar de 34.50% al 36.20%."

In [ ]:
texto = """
Durante el trimestre enero-marzo de 2026, los principales indicadores de desempeño tuvieron el siguiente comportamiento con respecto al mismo período de 2025:

<ul style="padding-left:20px; margin-top:10px;">

<li>El número de <b>inscritos</b> disminuyó en 11.08% al pasar de 641 a 570.</li>

<li>Las <b>cargas de materias</b> registraron una reducción de 5.93% al disminuir de 16,854 a 15,854.</li>

<li>La <b>eficiencia terminal</b> tuvo un aumento significativo del 19.05%, pasando del 25.51% al 30.37%.</li>

<li>Finalmente, la <b>tasa de deserción</b> incrementó en 4.93% al pasar de 34.50% al 36.20%.</li>

</ul>
"""

In [ ]:
analisis_formateado = analisis_final.replace(". ", ".<br><br>")
analisis_formateado

'En enero-marzo 2026, los principales indicadores académicos tuvieron el siguiente comportamiento:\n\nLos Inscritos disminuyeron en un 11.08%, pasando de 641 en enero-marzo 2025 a 570 en enero-marzo 2026.<br><br>Los Inscritos facturados experimentaron un aumento del 2.95%, ascendiendo de 542 a 558.<br><br>Las Cargas de materias mostraron una reducción del 5.93%, pasando de 16,854 a 15,854.<br><br>La Eficiencia terminal se incrementó en un 19.05%, situándose en 30.37% frente al 25.51% del periodo anterior.<br><br>Finalmente, la Tasa de Deserción registró un aumento del 4.93%, pasando del 34.50% al 36.20%.'

### Preparar para envío

In [ ]:
tabla_html = acumulado.to_html(index=False, border=0)

In [ ]:
tabla_html = tabla_html.replace(
    "<table",
    '<table style="border-collapse:collapse; width:100%; font-size:14px;"'
).replace(
    "<th>",
    '<th style="border:1px solid #ddd; padding:8px; background:#0033A0; color:white; text-align:center;">'
).replace(
    "<td>",
    '<td style="border:1px solid #ddd; padding:8px; text-align:center;">'
)

In [ ]:
logo_url="https://erp.agcollege.com.mx/assets/img/SIIMAG_Blue.png"
logo_url="https://raw.githubusercontent.com/claudiodanielpc-ag/siimag/refs/heads/main/logo_siimag_nuevo_transp.png"

In [ ]:
html = f"""
<html>
<body style="margin:0; padding:0; background:#f5f7fa; font-family:'Century Gothic', Arial, sans-serif;">

<table width="100%" cellpadding="0" cellspacing="0" style="background:#f5f7fa; padding:20px;">
<tr>
<td align="center">

<table width="700" cellpadding="0" cellspacing="0" style="background:white; border-radius:8px;">

    <!-- HEADER -->
    <tr>
        <td style="
            background:#ffffff;
            padding:25px 20px 20px 20px;
            text-align:center;
        ">

            <!-- LOGO -->
            <img src="{logo_url}"
                 style="
                    height:85px;
                    width:auto;
                    display:block;
                    margin:0 auto 10px auto;
                 ">

            <!-- TÍTULO -->
            <h2 style="
                color:#0033A0;
                margin:10px 0 5px 0;
                font-size:34px;
                font-weight:bold;
            ">
                Pulso Académico SIIMAG
            </h2>

            <!-- SUBTÍTULO -->
            <p style="
                color:#666;
                font-size:16px;
                margin:0;
            ">
                Enero – Marzo 2026
            </p>

        </td>
    </tr>

    <!-- TEXTO -->
    <tr>
        <td style="padding:25px; color:#333;">
            <p style="
                text-align:justify;
                line-height:1.7;
                font-size:14px;
                margin:0;
            ">
                {texto}
            </p>
        </td>
    </tr>

    <!-- TABLA -->
    <tr>
        <td style="padding:0 25px 25px 25px;">
            <h3 style="
                color:#0033A0;
                margin-bottom:10px;
            ">
                Detalle de indicadores
            </h3>

            {tabla_html}
        </td>
    </tr>

    <!-- CTA -->
    <tr>
        <td style="padding:30px; text-align:center;">

            <p style="
                font-size:16px;
                color:#333;
                margin-bottom:20px;
            ">
                <strong>Consulta el detalle completo en el SIIMAG</strong>
            </p>

            <a href="https://erp.agcollege.com.mx/#/login"
               style="
               background-color:#0033A0;
               color:white;
               padding:12px 25px;
               text-decoration:none;
               border-radius:5px;
               display:inline-block;
               font-weight:bold;
               font-size:14px;
               ">
               Ir al SIIMAG
            </a>

        </td>
    </tr>

    <!-- FOOTER -->
    <tr>
        <td style="
            padding:15px;
            text-align:center;
            font-size:12px;
            color:#777;
        ">
            Sistema de Información, Inteligencia y Monitoreo de Academia Global (SIIMAG)
        </td>
    </tr>

</table>

</td>
</tr>
</table>

</body>
</html>
"""

In [ ]:
#lista_correos=[nelson,karla,jazmín, javier,rosy verduzco, rodolfo castro, elsie, vanessa, andreyely, fer castro, nery, ernesto, ana belén, erik, claudio, almendra, abdiel, lety, fedra]
lista_correos= ["nelson.amparan@academiaglobal.mx",
                "karla.garzon@academiaglobal.mx",
                "jazmin.garzon@academiaglobal.mx",
                "javier.cazarez@academiaglobal.mx",
                "rosario.verduzco@academiaglobal.mx",
                "rodolfo.castro@academiaglobal.mx",
                "elsie.garzon@academiaglobal.mx",
                "vanessa.fuentes@academiaglobal.mx",
                "andreyely.ronquillo@academiaglobal.mx",
                "fernanda.castro@academiaglobal.mx",
                "nery.perez@academiaglobal.mx",
                "ernesto.torres@academiaglobal.mx",
                "anabelen.avila@academiaglobal.mx",
                "erik.velasco@academiaglobal.mx",
                "claudio.pacheco@academiaglobal.mx",
                "almendra.navidad@academiaglobal.mx",
                "abdiel.gutierrez@academiaglobal.mx",
                "leticia.beltran@academiaglobal.mx",
                "fedra.llanes@academiaglobal.mx",
                "analaura.roman@academiaglobal.mx",
                "mayra.alvarez@academiaglobal.mx"
                ]

In [ ]:
#Envío
msg = MIMEMultipart("alternative")
msg["Subject"] = "Pulso Académico SIIMAG enero-marzo 2026"
msg["From"] = "siimag@academiaglobal.mx"
msg["Bcc"] = ", ".join(lista_correos)

msg.attach(MIMEText(html, "html"))

with smtplib.SMTP("smtp.gmail.com", 587) as server:
    server.starttls()
    server.login("siimag@academiaglobal.mx", "oigp hpxu immp gsks")
    server.send_message(msg)